# Notebook 08 — Linear Probes (Rebuilt for NB4 v2 + NB7 TPU)

**Research question (RQ1):** Do the models' hidden representations encode action reversibility in a linearly decodable form?

This notebook uses only:
- NB4 v2 dataset: 2,001 integrated records, with the 2,000-record primary Sandbox subset used for the main analysis.
- NB4 v2 grouped splits: seeds 0, 1 and 2.
- NB7 TPU extraction runs:
  - Qwen: `run_20261123_tpuv5e4_qwen_01`
  - Llama: `run_20261123_tpuv5e4_llama_01`

The old 2,052-record legacy dataset and legacy hidden-state folders are deliberately not used.

## Methodological safeguards
- Split membership is taken from the frozen NB4 v2 grouped split file.
- Scenario groups are checked for train/test leakage.
- L2 logistic regression uses a **pre-specified C=1.0**. No hyperparameter or layer selection is performed using test performance.
- Every hidden-state index is evaluated; layer 0 is the embedding output.
- The two NB7 representations are evaluated separately: `last_action_token` and `mean_action_tokens`.
- Binary evaluation uses only reversible vs irreversible records; partially reversible records are excluded from that binary task, not silently relabelled.
- Three-class evaluation retains all three labels.
- ScienceWorld has only one labelled integrated record and is excluded from the primary Sandbox analysis.

Run the notebook top-to-bottom. Start with the configuration and integrity checks. The full probe sweep is an explicit later step and is checkpointed after every layer.

In [ ]:
import os

THREAD_LIMIT = "4"

for var in [
    "OMP_NUM_THREADS",
    "MKL_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[var] = THREAD_LIMIT

from threadpoolctl import threadpool_limits, threadpool_info

threadpool_limits(limits=4)

print("Thread configuration:")
for pool in threadpool_info():
    print(pool["internal_api"], pool["num_threads"])

print("\nNB8 CPU thread limits applied.")

## 1. Runtime and project paths

Run this notebook in the `ADNN TPU (Python 3.11)` Jupyter kernel on the TPU VM. The probes themselves run on CPU; the TPU is not needed after NB7 extraction.

In [ ]:
from pathlib import Path
import os, json, hashlib, platform, sys, time, math, csv
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    roc_auc_score, confusion_matrix
)

PROJECT_DIR = Path.home() / "adnn"
DATASET_PATH = PROJECT_DIR / "input" / "NB4_v2" / "integrated_dataset_v2.jsonl"
SPLITS_PATH = PROJECT_DIR / "input" / "NB4_v2" / "grouped_splits_v2.json"

OUTPUT_ROOT = PROJECT_DIR / "outputs" / "NB8_linear_probes"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = {
    "qwen": {
        "name": "Qwen/Qwen2.5-7B-Instruct",
        "directory": PROJECT_DIR / "outputs" / "NB7_TPUv5e4" / "qwen25_7b" / "run_20261123_tpuv5e4_qwen_01",
        "hidden_size": 3584,
        "hidden_state_count": 29,
        "expected_shards": 251,
    },
    "llama": {
        "name": "meta-llama/Llama-3.1-8B-Instruct",
        "directory": PROJECT_DIR / "outputs" / "NB7_TPUv5e4" / "llama31_8b" / "run_20261123_tpuv5e4_llama_01",
        "hidden_size": 4096,
        "hidden_state_count": 33,
        "expected_shards": 251,
    },
}

EXPECTED_RECORDS = 2001
EXPECTED_DATASET_SHA256 = "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd"

SEEDS = [0, 1, 2]
HORIZONS = ["1", "2", "3"]
LABELS_3CLASS = ["reversible", "partially_reversible", "irreversible"]
LABELS_BINARY = ["reversible", "irreversible"]
REPRESENTATIONS = ["last_action_token", "mean_action_tokens"]

# Pre-specified regularisation; do not tune this against test performance.
LOGISTIC_REGRESSION_C = 1.0
MAX_ITER = 1500
RANDOM_STATE = 42

# Full sweep defaults. You can narrow these for a pilot before the full run.
RUN_MODELS = ["qwen", "llama"]
RUN_REPRESENTATIONS = REPRESENTATIONS
RUN_HORIZONS = HORIZONS
RUN_SEEDS = SEEDS
RUN_LAYER_INDICES = None  # None = every hidden-state index for each model

print("Python:", sys.version)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("Dataset:", DATASET_PATH, DATASET_PATH.exists())
print("Splits:", SPLITS_PATH, SPLITS_PATH.exists())
print("Output:", OUTPUT_ROOT)

## 2. Load and fingerprint the NB4 v2 dataset

The expected dataset hash is frozen from the successful NB4 v2 run. This check prevents accidental use of the legacy dataset or a modified copy.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

assert DATASET_PATH.is_file(), f"Missing dataset: {DATASET_PATH}"
dataset_sha256 = sha256_file(DATASET_PATH)
print("Dataset SHA-256:", dataset_sha256)
assert dataset_sha256 == EXPECTED_DATASET_SHA256, "Dataset hash mismatch: stop; do not proceed."

records = []
with DATASET_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

assert len(records) == EXPECTED_RECORDS, (len(records), EXPECTED_RECORDS)
all_ids = [r["id"] for r in records]
assert len(all_ids) == len(set(all_ids)), "Duplicate dataset record IDs."

primary = [r for r in records if r.get("dataset_role") == "primary_sandbox"]
non_primary = [r for r in records if r.get("dataset_role") != "primary_sandbox"]
primary_ids = [r["id"] for r in primary]
primary_id_set = set(primary_ids)

assert len(primary) == 2000, f"Expected 2,000 primary Sandbox records; got {len(primary)}"
assert len(non_primary) == 1, f"Expected one non-primary record; got {len(non_primary)}"

for r in primary:
    assert r.get("label") in LABELS_3CLASS, (r["id"], r.get("label"))
    labels_by_horizon = r.get("metadata", {}).get("labels_by_horizon", {})
    for h in HORIZONS:
        assert labels_by_horizon.get(h) in LABELS_3CLASS, (r["id"], h, labels_by_horizon.get(h))

print("Integrated records:", len(records))
print("Primary Sandbox records:", len(primary))
print("Non-primary records (excluded from primary analysis):", len(non_primary))
print("Primary class distribution:", dict(Counter(r["label"] for r in primary)))
print("Dataset fingerprint verified.")

## 3. Load and validate the frozen grouped splits

`grouped_splits_v2.json` stores train and test IDs separately for each seed. The notebook does not create replacement splits.

In [ ]:
with SPLITS_PATH.open("r", encoding="utf-8") as f:
    split_payload = json.load(f)

assert split_payload.get("dataset_sha256") == dataset_sha256, "Split file is not bound to this dataset."
assert split_payload.get("split_method") == "StratifiedGroupKFold_v2", split_payload.get("split_method")
split_map = split_payload["splits"]
assert set(map(str, SEEDS)).issubset(set(split_map)), f"Missing seeds: {SEEDS}"

record_by_id = {r["id"]: r for r in primary}
SPLITS = {}

for seed in SEEDS:
    item = split_map[str(seed)]
    train_ids = list(item["train_ids"])
    test_ids = list(item["test_ids"])
    train_set, test_set = set(train_ids), set(test_ids)

    assert len(train_ids) == len(train_set), f"Seed {seed}: duplicate train IDs"
    assert len(test_ids) == len(test_set), f"Seed {seed}: duplicate test IDs"
    assert train_set.isdisjoint(test_set), f"Seed {seed}: train/test ID overlap"
    assert train_set | test_set == primary_id_set, f"Seed {seed}: split does not cover primary IDs"
    assert train_set <= primary_id_set and test_set <= primary_id_set

    train_scenarios = {record_by_id[i].get("scenario_id") for i in train_ids}
    test_scenarios = {record_by_id[i].get("scenario_id") for i in test_ids}
    scenario_overlap = train_scenarios & test_scenarios
    assert not scenario_overlap, f"Seed {seed}: scenario leakage ({len(scenario_overlap)})"

    train_pairs = {record_by_id[i].get("pair_id") for i in train_ids if record_by_id[i].get("pair_id") is not None}
    test_pairs = {record_by_id[i].get("pair_id") for i in test_ids if record_by_id[i].get("pair_id") is not None}
    pair_overlap = train_pairs & test_pairs
    assert not pair_overlap, f"Seed {seed}: pair leakage ({len(pair_overlap)})"

    SPLITS[seed] = {"train_ids": train_ids, "test_ids": test_ids}
    print(f"Seed {seed}: train={len(train_ids)}, test={len(test_ids)}, scenario overlap=0, pair overlap=0")

print("All frozen grouped splits validated.")

## 4. Validate NB7 manifests and shard inventories

The run-specific folders below are the authoritative fresh NB7 outputs. This cell checks the final audit summary, dataset binding, shard count, and model dimensions before indexing any arrays.

In [ ]:
def find_values_for_key(obj, wanted_key):
    found = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == wanted_key:
                found.append(v)
            found.extend(find_values_for_key(v, wanted_key))
    elif isinstance(obj, list):
        for v in obj:
            found.extend(find_values_for_key(v, wanted_key))
    return found

MODEL_SHARDS = {}

for model_key, cfg in MODEL_CONFIGS.items():
    model_dir = cfg["directory"]
    manifest_path = model_dir / "extraction_manifest.json"
    shards_dir = model_dir / "shards"
    assert manifest_path.is_file(), f"{model_key}: missing manifest {manifest_path}"
    assert shards_dir.is_dir(), f"{model_key}: missing shards directory {shards_dir}"

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    assert manifest.get("status") == "completed", (model_key, manifest.get("status"))

    final_audits = find_values_for_key(manifest, "final_audit")
    assert final_audits, f"{model_key}: final_audit missing from manifest"
    audit = final_audits[-1]
    assert audit["shards_checked"] == cfg["expected_shards"], (model_key, audit)
    assert audit["shards_passed"] == cfg["expected_shards"], (model_key, audit)
    assert audit["records_recovered"] == EXPECTED_RECORDS, (model_key, audit)
    assert audit["ordered_record_ids_match"] is True, (model_key, audit)
    assert audit["dataset_sha256"] == dataset_sha256, (model_key, audit["dataset_sha256"])

    shard_files = sorted(shards_dir.glob("shard_*.npz"))
    assert len(shard_files) == cfg["expected_shards"], (model_key, len(shard_files))
    MODEL_SHARDS[model_key] = shard_files

    print(f"\n{cfg['name']}")
    print("  Run directory:", model_dir)
    print("  Status:", manifest["status"])
    print("  Shards:", len(shard_files))
    print("  Final audit:", audit)

print("\nBoth fresh NB7 runs passed manifest-level checks.")

## 5. Index and validate every shard

This is a read-only pass. It verifies unique record IDs, array names, shapes, FP32 dtype and finite values. It does not modify the NB7 files.

In [ ]:
REPRESENTATION_KEYS = ["last_action_token", "mean_action_tokens"]
SHARD_INDEX = {}

for model_key, cfg in MODEL_CONFIGS.items():
    index = {}
    total_rows = 0

    for shard_path in MODEL_SHARDS[model_key]:
        with np.load(shard_path, allow_pickle=False) as z:
            required = {"record_ids", *REPRESENTATION_KEYS}
            missing_keys = required - set(z.files)
            assert not missing_keys, (shard_path.name, missing_keys)

            shard_ids = [str(x) for x in z["record_ids"].tolist()]
            n = len(shard_ids)
            total_rows += n

            for rep in REPRESENTATION_KEYS:
                arr = z[rep]
                expected = (n, cfg["hidden_state_count"], cfg["hidden_size"])
                assert arr.shape == expected, (model_key, shard_path.name, rep, arr.shape, expected)
                assert arr.dtype == np.float32, (model_key, shard_path.name, rep, arr.dtype)
                assert np.isfinite(arr).all(), (model_key, shard_path.name, rep, "non-finite values")

            for row_idx, rid in enumerate(shard_ids):
                assert rid not in index, f"{model_key}: duplicate ID {rid}"
                index[rid] = (shard_path, row_idx)

    assert len(index) == EXPECTED_RECORDS, (model_key, len(index))
    assert set(index) == set(all_ids), f"{model_key}: shard IDs do not match dataset IDs"
    SHARD_INDEX[model_key] = index

    print(f"{cfg['name']}: {len(MODEL_SHARDS[model_key])} shards, {total_rows} rows, {len(index)} unique IDs; all arrays validated.")

print("Shard-level structural validation passed for both models.")

## 6. Layer-wise representation loader

Loads only the requested layer and records, rather than materialising all hidden states in RAM.

In [ ]:
def load_layer_representations(model_key, record_ids, layer_idx, representation):
    cfg = MODEL_CONFIGS[model_key]
    if representation not in REPRESENTATION_KEYS:
        raise ValueError(representation)
    if not 0 <= layer_idx < cfg["hidden_state_count"]:
        raise ValueError(f"Invalid layer {layer_idx} for {model_key}")

    record_ids = list(record_ids)
    missing = [rid for rid in record_ids if rid not in SHARD_INDEX[model_key]]
    if missing:
        raise KeyError(f"{model_key}: missing IDs: {missing[:5]}")

    grouped = defaultdict(list)
    for output_row, rid in enumerate(record_ids):
        shard_path, shard_row = SHARD_INDEX[model_key][rid]
        grouped[shard_path].append((output_row, shard_row))

    X = np.empty((len(record_ids), cfg["hidden_size"]), dtype=np.float32)
    for shard_path, locations in grouped.items():
        with np.load(shard_path, allow_pickle=False) as z:
            layer_values = z[representation][:, layer_idx, :]
            for output_row, shard_row in locations:
                X[output_row] = layer_values[shard_row]

    if not np.isfinite(X).all():
        raise ValueError("Non-finite representation values.")
    return X

# Smoke-test one record, layer 0 and final layer for both representations/models.
example_id = primary_ids[0]
for model_key, cfg in MODEL_CONFIGS.items():
    for rep in REPRESENTATION_KEYS:
        for layer in sorted({0, cfg["hidden_state_count"] - 1}):
            x = load_layer_representations(model_key, [example_id], layer, rep)
            assert x.shape == (1, cfg["hidden_size"])
            assert x.dtype == np.float32 and np.isfinite(x).all()
            print(model_key, rep, "layer", layer, "->", x.shape, x.dtype)
print("Representation loader smoke tests passed.")

## 7. Label construction and probe functions

The primary analysis is Sandbox-only. Labels are obtained from `metadata.labels_by_horizon`, not from model inputs or hidden-state metadata.

In [ ]:
def labels_for(records_subset, horizon, task):
    labels = [r["metadata"]["labels_by_horizon"][str(horizon)] for r in records_subset]
    if task == "3class":
        return np.asarray(labels, dtype=object), LABELS_3CLASS
    if task == "binary":
        keep = np.asarray([y in LABELS_BINARY for y in labels], dtype=bool)
        return np.asarray(labels, dtype=object), LABELS_BINARY, keep
    raise ValueError(task)

def make_probe():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            penalty="l2",
            C=LOGISTIC_REGRESSION_C,
            solver="lbfgs",
            class_weight="balanced",
            max_iter=MAX_ITER,
            random_state=RANDOM_STATE,
        )),
    ])

def evaluate_predictions(y_true, y_pred, y_prob, classes, labels):
    result = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
    }
    if len(classes) == 2:
        positive_label = LABELS_BINARY[1]
        positive_idx = list(classes).index(positive_label)
        result["auroc"] = float(roc_auc_score((y_true == positive_label).astype(int), y_prob[:, positive_idx]))
    else:
        result["auroc"] = float(roc_auc_score(y_true, y_prob, labels=list(classes), multi_class="ovr", average="macro"))
    return result

# Synthetic smoke test: validates the sklearn pipeline without touching research data.
rng = np.random.default_rng(42)
X_demo = rng.normal(size=(120, 12))
y_demo = np.array(["reversible"] * 60 + ["irreversible"] * 60, dtype=object)
demo = make_probe()
demo.fit(X_demo[:90], y_demo[:90])
pred = demo.predict(X_demo[90:])
prob = demo.predict_proba(X_demo[90:])
demo_metrics = evaluate_predictions(y_demo[90:], pred, prob, demo.classes_, LABELS_BINARY)
assert all(np.isfinite(list(demo_metrics.values())))
print("Synthetic probe smoke test:", demo_metrics)

## 8. Pilot on one prespecified configuration

Run this before the full sweep. It evaluates Qwen, last-action-token, H3, final hidden-state index, seed 0. It is a pipeline check, not a basis for selecting the best layer or reporting a final result.

In [ ]:
PILOT_MODEL = "qwen"
PILOT_REPRESENTATION = "last_action_token"
PILOT_HORIZON = "3"
PILOT_LAYER = MODEL_CONFIGS[PILOT_MODEL]["hidden_state_count"] - 1
PILOT_SEED = 0

train_ids = SPLITS[PILOT_SEED]["train_ids"]
test_ids = SPLITS[PILOT_SEED]["test_ids"]
train_records = [record_by_id[i] for i in train_ids]
test_records = [record_by_id[i] for i in test_ids]

y_train_all = np.asarray([r["metadata"]["labels_by_horizon"][PILOT_HORIZON] for r in train_records], dtype=object)
y_test_all = np.asarray([r["metadata"]["labels_by_horizon"][PILOT_HORIZON] for r in test_records], dtype=object)

X_train_all = load_layer_representations(PILOT_MODEL, train_ids, PILOT_LAYER, PILOT_REPRESENTATION)
X_test_all = load_layer_representations(PILOT_MODEL, test_ids, PILOT_LAYER, PILOT_REPRESENTATION)

train_keep = np.isin(y_train_all, LABELS_BINARY)
test_keep = np.isin(y_test_all, LABELS_BINARY)
Xtr, ytr = X_train_all[train_keep], y_train_all[train_keep]
Xte, yte = X_test_all[test_keep], y_test_all[test_keep]

pilot_model = make_probe()
pilot_model.fit(Xtr, ytr)
pilot_pred = pilot_model.predict(Xte)
pilot_prob = pilot_model.predict_proba(Xte)
PILOT_METRICS = evaluate_predictions(yte, pilot_pred, pilot_prob, pilot_model.classes_, LABELS_BINARY)

print("Pilot configuration:", PILOT_MODEL, PILOT_REPRESENTATION, "H"+PILOT_HORIZON, "layer", PILOT_LAYER, "seed", PILOT_SEED)
print("Train rows (binary):", len(ytr), "Test rows (binary):", len(yte))
print("Metrics:", PILOT_METRICS)
assert len(ytr) > 0 and len(yte) > 0

## 9. Full layer-wise experiment sweep

The sweep evaluates both models, both representation types, all hidden-state indices, all three horizons, all three split seeds, and both classification tasks.

**Important:** the test split is used only for the final score of each prespecified configuration. There is no best-layer selection, hyperparameter tuning, or threshold tuning based on test performance.

Results are checkpointed to `results.csv` after each layer. If the notebook disconnects, rerun this cell; completed configuration keys are skipped.

In [ ]:
from time import perf_counter
import numpy as np

# Refit the existing sklearn pipeline and time it.
cpu_probe = make_probe()

t0 = perf_counter()
cpu_probe.fit(Xtr, ytr)
cpu_seconds = perf_counter() - t0

cpu_pred = cpu_probe.predict(Xte)
cpu_prob = cpu_probe.predict_proba(Xte)

cpu_metrics = evaluate_predictions(
    yte, cpu_pred, cpu_prob, cpu_probe.classes_, LABELS_BINARY
)

benchmark_path = OUTPUT_ROOT / "nb8_jax_benchmark_qwen_h3_seed0.npz"

np.savez_compressed(
    benchmark_path,
    X_train=Xtr.astype(np.float32),
    y_train=ytr.astype(str),
    X_test=Xte.astype(np.float32),
    y_test=yte.astype(str),
    classes=cpu_probe.classes_.astype(str),
    cpu_pred=cpu_pred.astype(str),
    cpu_prob=cpu_prob.astype(np.float64),
    cpu_seconds=np.asarray(cpu_seconds),
    C=np.asarray(LOGISTIC_REGRESSION_C),
)

print("CPU benchmark exported.")
print("Training shape:", Xtr.shape)
print("Testing shape:", Xte.shape)
print("CPU training seconds:", round(cpu_seconds, 3))
print("CPU metrics:", cpu_metrics)
print("Saved:", benchmark_path)

In [ ]:
RESULTS_CSV = OUTPUT_ROOT / "nb8_probe_results.csv"
RUN_CONFIG_JSON = OUTPUT_ROOT / "nb8_run_config.json"

run_config = {
    "dataset_sha256": dataset_sha256,
    "split_method": split_payload.get("split_method"),
    "seeds": RUN_SEEDS,
    "horizons": RUN_HORIZONS,
    "models": RUN_MODELS,
    "representations": RUN_REPRESENTATIONS,
    "layer_indices": "all" if RUN_LAYER_INDICES is None else RUN_LAYER_INDICES,
    "C": LOGISTIC_REGRESSION_C,
    "max_iter": MAX_ITER,
    "tasks": ["binary", "3class"],
    "selection_policy": "No test-set model/layer/hyperparameter selection; fixed C=1.0.",
}
RUN_CONFIG_JSON.write_text(json.dumps(run_config, indent=2), encoding="utf-8")

if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    completed_keys = set(results_df["result_key"].astype(str))
    results_rows = results_df.to_dict("records")
else:
    completed_keys = set()
    results_rows = []

def result_key(model, rep, layer, horizon, seed, task):
    return "|".join(map(str, [model, rep, layer, horizon, seed, task]))

total_configs = sum(
    len(range(MODEL_CONFIGS[m]["hidden_state_count"])) if RUN_LAYER_INDICES is None else len(RUN_LAYER_INDICES)
    for m in RUN_MODELS
) * len(RUN_REPRESENTATIONS) * len(RUN_HORIZONS) * len(RUN_SEEDS) * 2

print("Configurations in requested sweep:", total_configs)
print("Already completed:", len(completed_keys))
start_time = time.time()

for model_key in RUN_MODELS:
    cfg = MODEL_CONFIGS[model_key]
    layers = list(range(cfg["hidden_state_count"])) if RUN_LAYER_INDICES is None else list(RUN_LAYER_INDICES)

    for rep in RUN_REPRESENTATIONS:
        for layer_idx in layers:
            layer_start = time.time()

            # Keep only this layer's matrices in memory.
            X_all = load_layer_representations(model_key, primary_ids, layer_idx, rep)
            row_by_id = {rid: i for i, rid in enumerate(primary_ids)}

            for horizon in RUN_HORIZONS:
                y_all = np.asarray(
                    [r["metadata"]["labels_by_horizon"][str(horizon)] for r in primary],
                    dtype=object
                )

                for seed in RUN_SEEDS:
                    train_ids = SPLITS[seed]["train_ids"]
                    test_ids = SPLITS[seed]["test_ids"]
                    train_idx = np.asarray([row_by_id[i] for i in train_ids], dtype=int)
                    test_idx = np.asarray([row_by_id[i] for i in test_ids], dtype=int)

                    for task in ["binary", "3class"]:
                        key = result_key(model_key, rep, layer_idx, horizon, seed, task)
                        if key in completed_keys:
                            continue

                        y_train = y_all[train_idx]
                        y_test = y_all[test_idx]
                        X_train = X_all[train_idx]
                        X_test = X_all[test_idx]

                        if task == "binary":
                            train_mask = np.isin(y_train, LABELS_BINARY)
                            test_mask = np.isin(y_test, LABELS_BINARY)
                            X_train, y_train = X_train[train_mask], y_train[train_mask]
                            X_test, y_test = X_test[test_mask], y_test[test_mask]
                            labels = LABELS_BINARY
                        else:
                            labels = LABELS_3CLASS

                        if len(set(y_train)) != len(labels) or len(set(y_test)) != len(labels):
                            raise ValueError(f"Missing class in train/test: {key}; train={Counter(y_train)}, test={Counter(y_test)}")

                        probe = make_probe()
                        probe.fit(X_train, y_train)
                        y_pred = probe.predict(X_test)
                        y_prob = probe.predict_proba(X_test)
                        metrics = evaluate_predictions(y_test, y_pred, y_prob, probe.classes_, labels)

                        row = {
                            "result_key": key,
                            "model": model_key,
                            "model_name": cfg["name"],
                            "representation": rep,
                            "layer_index": int(layer_idx),
                            "horizon": str(horizon),
                            "seed": int(seed),
                            "task": task,
                            "n_train": int(len(y_train)),
                            "n_test": int(len(y_test)),
                            "C": float(LOGISTIC_REGRESSION_C),
                            **metrics,
                        }
                        results_rows.append(row)
                        completed_keys.add(key)

            pd.DataFrame(results_rows).to_csv(RESULTS_CSV, index=False)
            print(f"Checkpointed {model_key}/{rep}/layer_{layer_idx:02d} | total rows={len(results_rows)} | layer seconds={time.time()-layer_start:.1f}")

            del X_all
            import gc
            gc.collect()

print("Sweep finished in minutes:", round((time.time() - start_time) / 60, 2))
print("Results:", RESULTS_CSV)
print("Rows:", len(pd.DataFrame(results_rows)))

## 10. Aggregate results across split seeds

Report mean and standard deviation across the three frozen split seeds. Do not select the best layer using the test scores. Layer-wise curves are descriptive evaluations of the preregistered representation sweep.

In [ ]:
assert RESULTS_CSV.is_file(), "Run the full sweep first."
results = pd.read_csv(RESULTS_CSV)

expected_rows = sum(MODEL_CONFIGS[m]["hidden_state_count"] for m in RUN_MODELS) * len(RUN_REPRESENTATIONS) * len(RUN_HORIZONS) * len(RUN_SEEDS) * 2
print("Rows present:", len(results))
print("Expected rows for full default sweep:", expected_rows)
assert results["result_key"].is_unique, "Duplicate result keys."

aggregate = (
    results.groupby(["model", "representation", "layer_index", "horizon", "task"], as_index=False)
    .agg(
        balanced_accuracy_mean=("balanced_accuracy", "mean"),
        balanced_accuracy_sd=("balanced_accuracy", "std"),
        macro_f1_mean=("macro_f1", "mean"),
        macro_f1_sd=("macro_f1", "std"),
        accuracy_mean=("accuracy", "mean"),
        auroc_mean=("auroc", "mean"),
        n_seeds=("seed", "nunique"),
    )
)
AGGREGATE_CSV = OUTPUT_ROOT / "nb8_probe_aggregate.csv"
aggregate.to_csv(AGGREGATE_CSV, index=False)
display(aggregate.head(20))
print("Aggregate saved:", AGGREGATE_CSV)

## 11. Baselines and interpretation guardrails

The full sweep uses a class-balanced L2 logistic regression. A majority-class baseline is reported for each task/horizon/split so probe performance is not interpreted without a trivial reference.

In [ ]:
baseline_rows = []
for horizon in HORIZONS:
    for seed in SEEDS:
        train_records = [record_by_id[i] for i in SPLITS[seed]["train_ids"]]
        test_records = [record_by_id[i] for i in SPLITS[seed]["test_ids"]]
        ytr_all = np.asarray([r["metadata"]["labels_by_horizon"][horizon] for r in train_records], dtype=object)
        yte_all = np.asarray([r["metadata"]["labels_by_horizon"][horizon] for r in test_records], dtype=object)

        for task, labels in [("binary", LABELS_BINARY), ("3class", LABELS_3CLASS)]:
            if task == "binary":
                ytr = ytr_all[np.isin(ytr_all, LABELS_BINARY)]
                yte = yte_all[np.isin(yte_all, LABELS_BINARY)]
            else:
                ytr, yte = ytr_all, yte_all

            majority = Counter(ytr).most_common(1)[0][0]
            pred = np.full(len(yte), majority, dtype=object)
            baseline_rows.append({
                "horizon": horizon,
                "seed": seed,
                "task": task,
                "majority_class": majority,
                "accuracy": float(accuracy_score(yte, pred)),
                "balanced_accuracy": float(balanced_accuracy_score(yte, pred)),
                "macro_f1": float(f1_score(yte, pred, labels=labels, average="macro", zero_division=0)),
            })

baseline_df = pd.DataFrame(baseline_rows)
BASELINE_CSV = OUTPUT_ROOT / "nb8_majority_baselines.csv"
baseline_df.to_csv(BASELINE_CSV, index=False)
display(baseline_df)
print("Baselines saved:", BASELINE_CSV)

## 12. Layer-wise plots

These plots show all prespecified layers. They are descriptive and must not be used to claim an unbiased test-selected “best layer.”

In [ ]:
import matplotlib.pyplot as plt

for model_key in sorted(results["model"].unique()):
    for task in ["binary", "3class"]:
        subset = aggregate[(aggregate["model"] == model_key) & (aggregate["task"] == task)]
        if subset.empty:
            continue
        for horizon in HORIZONS:
            fig, ax = plt.subplots(figsize=(10, 5))
            h = subset[subset["horizon"].astype(str) == str(horizon)]
            for rep in REPRESENTATIONS:
                curve = h[h["representation"] == rep].sort_values("layer_index")
                if not curve.empty:
                    ax.plot(curve["layer_index"], curve["balanced_accuracy_mean"], marker="o", markersize=3, label=rep)
            ax.set_title(f"{model_key.upper()} — {task} — H{horizon}")
            ax.set_xlabel("Hidden-state index (0 = embedding output)")
            ax.set_ylabel("Mean test balanced accuracy across split seeds")
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.25)
            ax.legend()
            plt.show()

## 13. Final integrity and completion report

This cell verifies expected configuration coverage and writes a machine-readable completion report. If you ran only a pilot subset, it will explicitly report that the full sweep is incomplete.

In [ ]:
expected_full = {
    result_key(m, rep, layer, h, seed, task)
    for m in MODEL_CONFIGS
    for rep in REPRESENTATIONS
    for layer in range(MODEL_CONFIGS[m]["hidden_state_count"])
    for h in HORIZONS
    for seed in SEEDS
    for task in ["binary", "3class"]
}
observed = set(results["result_key"].astype(str))
missing = sorted(expected_full - observed)
unexpected = sorted(observed - expected_full)

completion = {
    "dataset_sha256": dataset_sha256,
    "integrated_records": len(records),
    "primary_sandbox_records": len(primary),
    "models": list(MODEL_CONFIGS),
    "representations": REPRESENTATIONS,
    "seeds": SEEDS,
    "horizons": HORIZONS,
    "expected_full_configurations": len(expected_full),
    "observed_configurations": len(observed),
    "missing_configurations": len(missing),
    "unexpected_configurations": len(unexpected),
    "full_sweep_complete": len(missing) == 0 and len(unexpected) == 0,
    "selection_policy": "Fixed C=1.0; no test-set layer or hyperparameter selection.",
    "results_csv": str(RESULTS_CSV),
    "aggregate_csv": str(AGGREGATE_CSV) if AGGREGATE_CSV.exists() else None,
}
COMPLETION_JSON = OUTPUT_ROOT / "nb8_completion_report.json"
COMPLETION_JSON.write_text(json.dumps(completion, indent=2), encoding="utf-8")
print(json.dumps(completion, indent=2))
if missing:
    print("First missing configurations:", missing[:10])

## 14. Backup NB8 results to both Google Drive roots

Run only after the results and completion report are saved. This copies the NB8 output folder to both existing Drive roots. It does not modify NB7.

In [ ]:
# Requires the existing rclone remote named `radha`.
# This cell is intentionally separate from the experiment run.
import subprocess

local_output = str(OUTPUT_ROOT)
destinations = [
    "radha:ir/NB8_linear_probes",
    "radha:ir_final/NB8_linear_probes",
]

for destination in destinations:
    print("\nBacking up to:", destination)
    subprocess.run(
        ["rclone", "copy", local_output, destination, "--progress", "--transfers", "2", "--checkers", "2"],
        check=True,
    )
    subprocess.run(
        ["rclone", "check", local_output, destination, "--one-way"],
        check=True,
    )

print("\nNB8 backup verified in both Drive roots.")